# DeepNowcast: Spatiotemporal Data Exploration (ERA5)

This notebook demonstrates how to load, visualize, and prepare multi-channel meteorological NetCDF tensors for the ConvLSTM network. We utilize `xarray` for handling the complex data structures.

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# Set plotting style
plt.style.use('dark_background')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Load the NetCDF Data
Assuming you have run `python scripts/fetch_era5.py` to download the sample.

In [ ]:
# Load the dataset
try:
    nc_file = '../data/raw/era5_india_2023_08_sample.nc'
    ds = xr.open_dataset(nc_file)
    print("Dataset successfully loaded!")
    display(ds)
except FileNotFoundError:
    print(f"File {nc_file} not found. Ensure you ran the fetch script or place a dummy file here.")

## 2. Visualize the Spatial Grids (Temperature & Precipitation)

In [ ]:
def plot_weather_snapshot(dataset, time_idx=0):
    """
    Plots a single snapshot in time for Temperature and Precipitation.
    Requires Cartopy for the coastline delineations.
    """
    fig, (ax1, ax2) = plt.subplots(1, 2, subplot_kw={'projection': ccrs.PlateCarree()}, figsize=(16, 6))
    
    # Time selected
    time_str = str(dataset.time[time_idx].values)[:16]
    fig.suptitle(f"Weather Snapshot over India: {time_str}", fontsize=16)
    
    # Plot Temperature (t2m - roughly mapped to Celsius)
    temp_c = dataset['t2m'][time_idx] - 273.15
    c1 = ax1.contourf(dataset.longitude, dataset.latitude, temp_c, cmap='inferno', transform=ccrs.PlateCarree())
    ax1.add_feature(cfeature.COASTLINE, edgecolor='white')
    ax1.set_title("2m Temperature (°C)")
    fig.colorbar(c1, ax=ax1, fraction=0.046, pad=0.04)
    
    # Plot Precipitation (tp)
    precip = dataset['tp'][time_idx]
    c2 = ax2.contourf(dataset.longitude, dataset.latitude, precip, cmap='Blues', transform=ccrs.PlateCarree())
    ax2.add_feature(cfeature.COASTLINE, edgecolor='black')
    ax2.set_title("Total Precipitation (m)")
    fig.colorbar(c2, ax=ax2, fraction=0.046, pad=0.04)
    
    plt.show()

# Uncomment to run if data is present:
# plot_weather_snapshot(ds, time_idx=10)

## 3. The 'Video Prediction' Concept
To train our ConvLSTM, we slice this sequence of frames into sub-sequences. 
We provide $T_{past} = 5$ frames (the input video), and the model learns to predict the next $T_{future} = 2$ frames.

In [ ]:
print("Preprocessing Strategy:")
print("1. Normalize the channels using Min-Max scaling to map values between 0 and 1.")
print("2. Extract underlying numpy arrays from xarray: `(Time, Channels, Lat, Lon)`")
print("3. Use PyTorch DataLoader to generate rolling windows (past 5 hours -> next 2 hours).")